In [ ]:
# [1] secrets
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

_s = UserSecretsClient()
for name in ("NGROK_AUTHTOKEN", "NGROK_DOMAIN", "GATEWAY_API_KEY", "HF_TOKEN"):
    value = _s.get_secret(name)
    if not value:
        raise RuntimeError(f"Kaggle secret {name} is missing (Add-ons > Secrets, attach to this notebook)")
    os.environ[name] = value.strip()
    print(name, "OK")
os.environ["NGROK_DOMAIN"] = os.environ["NGROK_DOMAIN"].removeprefix("https://").rstrip("/")
os.makedirs("/kaggle/working", exist_ok=True)
os.chdir("/kaggle/working")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok==7.2.3", "fastapi==0.115.6", "uvicorn==0.34.0", "httpx==0.28.1"], check=True)


In [ ]:
# [2] llama-server binary (dataset llamacpp-sm75-bin, else build from source ~26 min)
import glob, os, re, shutil, stat, subprocess

DATASET = "/kaggle/input/llamacpp-sm75-bin"
DEST = "/kaggle/working/llama-bin"


def find_server(root):
    hits = glob.glob(f"{root}/**/llama-server", recursive=True)
    return hits[0] if hits else None


if os.path.isdir(DATASET) and find_server(DATASET):
    shutil.copytree(DATASET, DEST, dirs_exist_ok=True, symlinks=True)
else:
    print("WARNING: dataset llamacpp-sm75-bin not attached -> building llama.cpp from source (~26 min).")
    print("Attach the dataset (see docs/runbook-kaggle.md) to skip this.")
    SRC = "/kaggle/working/llama.cpp"
    if not os.path.isdir(SRC):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", SRC], check=True)
    # libcuda: FindCUDAToolkit needs it at CONFIGURE time to create CUDA::cuda_driver, the linker at build time.
    # First hit wins: toolkit stubs (CUDA 13: targets/x86_64-linux/lib/stubs), toolkit compat, ldconfig, bounded find.
    # Linking the real driver/compat lib is fine (resolved dynamically at runtime). Private dir, build only.
    def _out(cmd, timeout):
        try:
            return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout).stdout
        except (subprocess.TimeoutExpired, OSError):
            return ""

    _ld = re.findall(r"^\s*libcuda\.so(?:\.1)?\s+\(.*\)\s*=>\s*(\S+)", _out(["ldconfig", "-p"], 30), re.M)
    _find = _out(["find", "/usr/lib", "/usr/lib64", "/lib", "/usr/local/nvidia", "/opt", "-name", "libcuda.so*"], 30).split()
    stub = None
    for name, paths in (("toolkit stubs", glob.glob("/usr/local/cuda*/**/stubs/libcuda.so*", recursive=True)),
                        ("toolkit compat", glob.glob("/usr/local/cuda*/compat/libcuda.so*")),
                        ("ldconfig", _ld), ("find", _find)):
        paths = sorted({os.path.realpath(x) for x in paths})
        print("libcuda candidates", name, ":", paths or "none")
        if paths and not stub:
            stub = next((x for x in paths if x.endswith("libcuda.so")), paths[0])
    print("chosen libcuda:", stub)
    if not stub:
        print(_out(["find", "/", "-xdev", "-name", "libcuda*"], 60))
        raise RuntimeError("libcuda not found: is the GPU accelerator on? Paste the lines above.")
    cfg = ["cmake", "-B", "build", "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75", "-DLLAMA_OPENSSL=OFF"]
    env = dict(os.environ)
    if stub:
        stub_dir = "/kaggle/working/cuda-stub"
        shutil.rmtree(stub_dir, ignore_errors=True)
        os.makedirs(stub_dir)
        for n in ("libcuda.so", "libcuda.so.1"):
            os.symlink(stub, f"{stub_dir}/{n}")
        env.update(LIBRARY_PATH=stub_dir, LD_LIBRARY_PATH=(stub_dir + ":" + os.environ.get("LD_LIBRARY_PATH", "")).rstrip(":"))
        cfg += [f"-DCMAKE_LIBRARY_PATH={stub_dir}", "-DCUDAToolkit_ROOT=/usr/local/cuda",
                f"-DCUDA_cuda_driver_LIBRARY={stub}", f"-DCUDA_DRIVER_LIBRARY={stub}"]
    subprocess.run(cfg, cwd=SRC, env=env, check=True)
    subprocess.run(["cmake", "--build", "build", "--target", "llama-server", "-j", str(os.cpu_count())],
                   cwd=SRC, env=env, check=True)
    shutil.copytree(f"{SRC}/build", DEST, dirs_exist_ok=True, symlinks=True)

LLAMA_SERVER = find_server(DEST)
assert LLAMA_SERVER, "llama-server not found after copy/build"
for root, _, files in os.walk(DEST):
    for f in files:
        p = os.path.join(root, f)
        os.chmod(p, os.stat(p).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
lib_dirs = sorted({os.path.dirname(p) for p in glob.glob(f"{DEST}/**/*.so*", recursive=True)} | {os.path.dirname(LLAMA_SERVER)})
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("llama-server:", LLAMA_SERVER)


In [ ]:
# [3] llama-server on GPU 0 (Qwen3-8B Q5_K_M, thinking off)
import glob, json, os, subprocess
from huggingface_hub import hf_hub_download

ggufs = sorted(glob.glob("/kaggle/input/qwen3-8b-gguf/**/*.gguf", recursive=True), key=lambda p: "Q5_K_M" not in p)
if ggufs:
    GGUF = ggufs[0]
else:
    print("dataset qwen3-8b-gguf not attached -> downloading from Hugging Face")
    GGUF = hf_hub_download("Qwen/Qwen3-8B-GGUF", "Qwen3-8B-Q5_K_M.gguf", local_dir="/kaggle/working/models",
                           token=os.environ["HF_TOKEN"])
os.environ["LLM_MODEL"] = os.path.basename(GGUF)
print("GGUF:", GGUF)

cmd = [
    LLAMA_SERVER, "-m", GGUF, "-ngl", "99", "-c", "16384", "-np", "2",
    "--jinja", "--reasoning-budget", "0",
    "--chat-template-kwargs", json.dumps({"enable_thinking": False}),
    "--host", "127.0.0.1", "--port", "8080",
]
os.makedirs("/kaggle/temp", exist_ok=True)  # ngoài /kaggle/working để log (có prompt) không nằm trong output lưu lại
llama_log = open("/kaggle/temp/llama.log", "wb")
llama_proc = subprocess.Popen(cmd, stdout=llama_log, stderr=subprocess.STDOUT,
                              env={k: os.environ[k] for k in ("PATH", "LD_LIBRARY_PATH", "HOME") if k in os.environ} | {"CUDA_VISIBLE_DEVICES": "0"})  # không đưa secret cho llama-server
print("llama-server pid", llama_proc.pid, "- log: /kaggle/temp/llama.log")


In [ ]:
%%writefile gateway.py
"""Authenticated gateway in front of llama-server + embedder + reranker (runs in a Kaggle notebook)."""

import hmac
import math
import os
import threading
import time
from contextlib import asynccontextmanager
from typing import Annotated, Protocol

import anyio
import httpx
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field, StringConstraints

MAX_ITEMS = 64
MAX_BODY = 1_000_000
Text = Annotated[str, StringConstraints(max_length=8000)]
BATCH = 32


class Embedder(Protocol):
    def encode(self, texts: list[str]) -> list[list[float]]: ...


class Reranker(Protocol):
    def predict(self, pairs: list[tuple[str, str]]) -> list[float]: ...  # raw logits


class EmbedReq(BaseModel):
    input: list[Text] = Field(min_length=1, max_length=MAX_ITEMS)
    model: str | None = None


class RerankReq(BaseModel):
    query: Text
    documents: list[Text] = Field(min_length=1, max_length=MAX_ITEMS)
    top_n: int | None = Field(default=None, ge=1)


def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x)) if x >= 0 else math.exp(x) / (1.0 + math.exp(x))


def create_app(
    embedder: Embedder | None,
    reranker: Reranker | None,
    llm_url: str,
    api_key: str,
    model_name: str = "",
) -> FastAPI:
    if not api_key:
        raise ValueError("api_key must not be empty")
    client = httpx.AsyncClient(timeout=httpx.Timeout(120.0, connect=5.0))

    @asynccontextmanager
    async def lifespan(_: FastAPI):
        yield
        await client.aclose()

    app = FastAPI(lifespan=lifespan, docs_url=None, redoc_url=None, openapi_url=None)
    llm_url = llm_url.rstrip("/")
    expected = f"Bearer {api_key}".encode()
    emb_lock, rer_lock = threading.Lock(), threading.Lock()

    class GuardMiddleware:
        """Pure ASGI: auth before any body is read, then cap body size."""

        def __init__(self, inner):
            self.inner = inner

        async def __call__(self, scope, receive, send):
            if scope["type"] != "http" or scope["path"] == "/health":
                return await self.inner(scope, receive, send)
            got = dict(scope["headers"]).get(b"authorization", b"")
            if not hmac.compare_digest(got, expected):
                return await JSONResponse({"detail": "unauthorized"}, status_code=401)(
                    scope, receive, send
                )
            declared = dict(scope["headers"]).get(b"content-length", b"0")
            if declared.isdigit() and int(declared) > MAX_BODY:
                return await JSONResponse(
                    {"detail": "body too large"}, status_code=413
                )(scope, receive, send)
            seen = 0

            async def capped():
                nonlocal seen
                msg = await receive()
                if msg["type"] == "http.request":
                    seen += len(msg.get("body", b""))
                    if seen > MAX_BODY:
                        raise HTTPException(status_code=413, detail="body too large")
                return msg

            await self.inner(scope, capped, send)

    app.add_middleware(GuardMiddleware)

    probe = (
        0.0,
        False,
    )  # (monotonic, llm ok): /health không xác thực, cache 5 s để không đốt quota ngrok / llama

    @app.get("/health")
    async def health():
        nonlocal probe
        if time.monotonic() - probe[0] > 5.0:
            try:
                r = await client.get(f"{llm_url}/health", timeout=2.0)
                ok = r.status_code == 200 and r.json().get("status") == "ok"
            except Exception:
                ok = False
            probe = (time.monotonic(), ok)
        llm = probe[1]
        emb, rer = embedder is not None, reranker is not None
        return {
            "status": "ok" if llm and emb and rer else "loading",
            "llm": llm,
            "embedder": emb,
            "reranker": rer,
        }

    @app.post("/v1/embeddings")
    def embeddings(req: EmbedReq):
        if embedder is None:
            raise HTTPException(status_code=503, detail="embedder unavailable")
        with emb_lock:
            vecs = embedder.encode(req.input)
        return {"data": [{"index": i, "embedding": v} for i, v in enumerate(vecs)]}

    @app.post("/v1/rerank")
    def rerank(req: RerankReq):
        if reranker is None:
            raise HTTPException(status_code=503, detail="reranker unavailable")
        with rer_lock:
            logits = reranker.predict([(req.query, d) for d in req.documents])
        scored = sorted(
            (
                {"index": i, "relevance_score": _sigmoid(float(s))}
                for i, s in enumerate(logits)
            ),
            key=lambda x: x["relevance_score"],
            reverse=True,
        )
        return {"results": scored[: req.top_n or len(scored)]}

    @app.post("/v1/chat/completions")
    async def chat(body: dict):
        url = f"{llm_url}/v1/chat/completions"
        try:
            if body.get("stream"):
                up = await client.send(
                    client.build_request("POST", url, json=body), stream=True
                )

                async def chunks():
                    try:
                        async for c in up.aiter_bytes():
                            yield c
                    finally:  # also runs on client disconnect (cancellation)
                        with anyio.CancelScope(shield=True):
                            await up.aclose()

                try:
                    return StreamingResponse(
                        chunks(),
                        status_code=up.status_code,
                        media_type="text/event-stream",
                    )
                except BaseException:
                    await up.aclose()
                    raise
            up = await client.post(url, json=body)
        except httpx.HTTPError:
            return JSONResponse({"detail": "llm unavailable"}, status_code=503)
        try:
            data = up.json()
        except ValueError:
            return JSONResponse({"detail": "llm bad response"}, status_code=502)
        return JSONResponse(data, status_code=up.status_code)

    return app


def load_models(device: str = "cuda:1"):
    """Load embedder + reranker (fp16). Imports are local so the module stays light for tests."""
    import torch
    from sentence_transformers import CrossEncoder, SentenceTransformer

    class _Emb:
        def __init__(self):
            self.m = SentenceTransformer(
                "AITeamVN/Vietnamese_Embedding",
                device=device,
                model_kwargs={"torch_dtype": torch.float16},
            )

        def encode(self, texts):
            return self.m.encode(
                texts, batch_size=BATCH, normalize_embeddings=True
            ).tolist()

    class _Rer:
        def __init__(self):
            self.m = CrossEncoder(
                "AITeamVN/Vietnamese_Reranker",
                device=device,
                activation_fn=torch.nn.Identity(),  # raw logits; gateway applies sigmoid
                model_kwargs={"torch_dtype": torch.float16},
            )

        def predict(self, pairs):
            return [float(x) for x in self.m.predict(pairs, batch_size=BATCH)]

    return _Emb(), _Rer()


def main() -> None:
    import uvicorn

    embedder, reranker = load_models()
    app = create_app(
        embedder,
        reranker,
        os.environ.get("LLM_URL", "http://127.0.0.1:8080"),
        os.environ["GATEWAY_API_KEY"],
        os.environ.get("LLM_MODEL", ""),
    )
    uvicorn.run(app, host="0.0.0.0", port=8000)


if __name__ == "__main__":
    main()


In [ ]:
# [5] gateway on :8000 (background thread; loads embedder + reranker fp16 on cuda:1)
import os, sys, threading, traceback
import uvicorn

sys.path.insert(0, os.getcwd())
import gateway

gateway_err = []


def _serve():
    try:
        embedder, reranker = gateway.load_models("cuda:1")
        app = gateway.create_app(embedder, reranker, "http://127.0.0.1:8080",
                                 os.environ["GATEWAY_API_KEY"], os.environ["LLM_MODEL"])
        uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")).run()
    except BaseException:
        gateway_err.append(traceback.format_exc())
        print(gateway_err[-1])


threading.Thread(target=_serve, daemon=True).start()
print("gateway thread started (models load while llama-server loads)")


In [ ]:
# [6] wait until gateway /health says ok (llm + embedder + reranker), max 15 min
import json, time, urllib.request

H = "http://127.0.0.1:8000/health"


def health():
    try:
        with urllib.request.urlopen(H, timeout=3) as r:
            return json.load(r)
    except Exception:
        return None


t0 = time.time()
while True:
    h = health()
    print(f"[{int(time.time() - t0):4d}s]", h or "gateway not up yet (loading models)")
    if h and h.get("status") == "ok":
        break
    if llama_proc.poll() is not None:
        print(open("/kaggle/temp/llama.log", errors="replace").read()[-3000:])
        raise RuntimeError(f"llama-server exited with code {llama_proc.returncode}")
    if gateway_err:
        raise RuntimeError("gateway thread crashed, see traceback above")
    if time.time() - t0 > 900:
        print(open("/kaggle/temp/llama.log", errors="replace").read()[-3000:])
        raise TimeoutError("not ready after 15 min")
    time.sleep(10)
print("local stack ready in", int(time.time() - t0), "s")


In [ ]:
# [7] public tunnel (static ngrok domain -> :8000)
import os, re
from pyngrok import ngrok

try:
    ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
    ngrok.connect(8000, "http", domain=os.environ["NGROK_DOMAIN"])
except Exception as e:
    # pyngrok puts the authtoken in its exception text: report only the type and the ERR_NGROK code
    code = re.search(r"ERR_NGROK_\d+", str(e).replace(os.environ["NGROK_AUTHTOKEN"], "***"))
    raise RuntimeError(f"ngrok setup failed: {type(e).__name__} {code.group() if code else ''}") from None
PUBLIC_URL = "https://" + os.environ["NGROK_DOMAIN"]


In [ ]:
# [8] smoke test through the gateway, then READY
import json, os, urllib.request

BASE = "http://127.0.0.1:8000"
AUTH = {"Authorization": "Bearer " + os.environ["GATEWAY_API_KEY"], "Content-Type": "application/json"}


def post(path, body):
    req = urllib.request.Request(BASE + path, json.dumps(body).encode(), AUTH)
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.load(r)


emb = post("/v1/embeddings", {"input": ["Thời gian thử việc tối đa là bao lâu?"]})
print("embeddings: dim =", len(emb["data"][0]["embedding"]))
rr = post("/v1/rerank", {"query": "thử việc tối đa bao lâu",
                         "documents": ["Thời gian thử việc không quá 60 ngày đối với công việc cần trình độ cao đẳng.",
                                       "Cách nấu phở bò truyền thống."]})
print("rerank:", rr["results"])
chat = post("/v1/chat/completions", {"model": os.environ["LLM_MODEL"], "max_tokens": 64, "temperature": 0,
                                     "messages": [{"role": "user", "content": "Xin chào! Trả lời một câu ngắn bằng tiếng Việt."}]})
msg = chat["choices"][0]["message"]["content"]
print("chat:", msg)
if "<think>" in msg:
    print("WARNING: thinking is NOT disabled; check llama-server flags")
print("READY", PUBLIC_URL)


In [ ]:
# [9] bounded keep-alive: log /health every 60 s for DEMO_HOURS, then shut everything down (Kaggle ToS rule 8: never run unattended).
# Still STOP THE SESSION yourself after the interview (Kaggle > Session options > Stop session) to release the GPU.
import json, time, urllib.request

DEMO_HOURS = 2
deadline = time.time() + DEMO_HOURS * 3600
while time.time() < deadline:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as r:
            print(time.strftime("%H:%M:%S"), json.load(r))
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "health failed:", e)
    time.sleep(60)
ngrok.kill()
llama_proc.terminate()
print("STOPPED: demo window over. Stop the session now (Session options > Stop session).")
